# PRISM-U — Stage 1 Controlled Benchmark

This notebook runs the complete first PRISM-U milestone in Google Colab:

1. Upload and extract the PRISM-U Stage 1 project.
2. Check the GPU.
3. Install dependencies.
4. Run the smoke test.
5. Run a **1-epoch end-to-end test**.
6. Clear the temporary test outputs.
7. Mount Google Drive.
8. Run the **real Stage 1 training**.
9. Back up checkpoints, splits, metrics, and probability files.
10. Inspect and download the final results.

### What Stage 1 creates

- `M_original`: ResNet-18 trained on the full target-training pool (`retain + forget`)
- `M_retrain`: a fresh ResNet-18 trained only on the retained set
- deterministic CIFAR-10 split
- checkpoints
- evaluation metrics
- per-example prediction probabilities

**Before starting:** in Colab choose **Runtime → Change runtime type → GPU**.


In [1]:
# STEP 1 — Check Colab GPU

import os
import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("\nWARNING: No GPU is enabled.")
    print("Go to Runtime → Change runtime type → GPU, then run this cell again.")


Python: 3.13.15
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
CUDA version: 13.0


In [2]:
# STEP 2 — Upload PRISM-U-stage1.zip

from google.colab import files
from pathlib import Path

zip_path = Path("/content/PRISM-U-stage1.zip")

if zip_path.exists():
    print("ZIP already exists:", zip_path)
else:
    uploaded = files.upload()
    if "PRISM-U-stage1.zip" not in uploaded:
        raise FileNotFoundError(
            "Please upload the file named exactly: PRISM-U-stage1.zip"
        )
    print("Uploaded:", zip_path)


Saving PRISM-U-stage1.zip to PRISM-U-stage1.zip
Uploaded: /content/PRISM-U-stage1.zip


In [3]:
# STEP 3 — Extract the project and enter its folder

import shutil
from pathlib import Path

project_dir = Path("/content/PRISM-U-stage1")

if project_dir.exists():
    print("Removing old extracted project...")
    shutil.rmtree(project_dir)

shutil.unpack_archive("/content/PRISM-U-stage1.zip", "/content")

if not project_dir.exists():
    raise FileNotFoundError("PRISM-U-stage1 folder was not found after extraction.")

os.chdir(project_dir)

print("Working directory:", os.getcwd())
print("\nTop-level files/folders:")
for p in sorted(project_dir.iterdir()):
    print(" -", p.name)


Working directory: /content/PRISM-U-stage1

Top-level files/folders:
 - .gitignore
 - README.md
 - checkpoints
 - configs
 - data
 - evaluation
 - experiments
 - models
 - requirements.txt
 - results
 - run_stage1_linux_mac.sh
 - run_stage1_windows.bat
 - splits
 - utils


In [4]:
# STEP 4 — Install dependencies

!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt

print("Dependencies installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 36.0 MB/s eta 0:00:00
Dependencies installed.


In [5]:
# STEP 5 — Verify project configuration and deterministic split

from pathlib import Path
import yaml
import json

config_path = Path("configs/cifar10_stage1.yaml")

with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

print("Configuration:")
print(json.dumps(cfg, indent=2))

split_path = Path("splits/cifar10_seed42.json")
print("\nExisting split file:", split_path.exists())

if split_path.exists():
    with open(split_path, "r") as f:
        split = json.load(f)

    for key in ["target_train", "forget", "retain", "auxiliary", "validation", "official_test"]:
        if key in split:
            print(f"{key:>14}: {len(split[key])}")


Configuration:
{
  "seed": 42,
  "data": {
    "root": "./data/cifar10",
    "split_file": "./splits/cifar10_seed42.json",
    "target_train_size": 40000,
    "auxiliary_size": 5000,
    "validation_size": 5000,
    "forget_fraction": 0.1,
    "num_workers": 2
  },
  "model": {
    "name": "resnet18_cifar",
    "num_classes": 10
  },
  "training": {
    "epochs": 40,
    "batch_size": 128,
    "learning_rate": 0.1,
    "momentum": 0.9,
    "weight_decay": 0.0005,
    "scheduler_milestones": [
      20,
      30
    ],
    "scheduler_gamma": 0.1,
    "label_smoothing": 0.0
  },
  "evaluation": {
    "batch_size": 256,
    "save_probabilities": true
  },
  "paths": {
    "checkpoints": "./checkpoints",
    "results": "./results"
  }
}

Existing split file: True
  target_train: 40000
        forget: 4000
        retain: 36000
     auxiliary: 5000
    validation: 5000
 official_test: 10000


In [6]:
# STEP 6 — Run the offline smoke test

!python -m experiments.smoke_test


SMOKE TEST PASSED
device=cuda, one-step loss=2.2336


## One-epoch plumbing test

This does **not** produce your real research models. It only checks that the full training/evaluation pipeline runs without crashing.

Both temporary models are trained for one epoch.


In [7]:
# STEP 7 — Run the 1-epoch end-to-end test

!python -m experiments.run_stage1 \
    --config configs/cifar10_stage1.yaml \
    --epochs 1



$ /usr/bin/python3 -m data.splits --config configs/cifar10_stage1.yaml
Split already exists: splits/cifar10_seed42.json
{'target_train': 40000, 'forget': 4000, 'retain': 36000, 'auxiliary': 5000, 'validation': 5000, 'official_test': 10000}

$ /usr/bin/python3 -m models.train_target --mode original --config configs/cifar10_stage1.yaml --epochs 1
100% 170M/170M [34:05<00:00, 83.3kB/s]
Training mode=original split=target_train samples=40000 device=cuda
epoch 001/1 train_loss=1.9226 train_acc=0.3027 val_loss=1.9664 val_acc=0.3160
Saved checkpoint: checkpoints/original_seed42.pt
Final test accuracy: 0.3257

$ /usr/bin/python3 -m models.train_target --mode retrain --config configs/cifar10_stage1.yaml --epochs 1
Training mode=retrain split=retain samples=36000 device=cuda
epoch 001/1 train_loss=2.1301 train_acc=0.2555 val_loss=1.8053 val_acc=0.3222
Saved checkpoint: checkpoints/retrain_seed42.pt
Final test accuracy: 0.3354

$ /usr/bin/python3 -m evaluation.evaluate --config configs/cifar10_s

In [8]:
# STEP 8 — Inspect the one-epoch outputs

from pathlib import Path
import json

print("CHECKPOINTS")
for p in sorted(Path("checkpoints").glob("*")):
    print(" -", p)

print("\nRESULT FILES")
for p in sorted(Path("results").glob("*")):
    print(" -", p)

for name in ["eval_original_seed42.json", "eval_retrain_seed42.json"]:
    path = Path("results") / name
    if path.exists():
        print("\n", "=" * 70)
        print(name)
        with open(path, "r") as f:
            data = json.load(f)
        print(json.dumps(data, indent=2))


CHECKPOINTS
 - checkpoints/original_seed42.pt
 - checkpoints/retrain_seed42.pt

RESULT FILES
 - results/eval_original_seed42.json
 - results/eval_retrain_seed42.json
 - results/probs_original_seed42.npz
 - results/probs_retrain_seed42.npz
 - results/train_original_seed42.json
 - results/train_retrain_seed42.json

eval_original_seed42.json
{
  "checkpoint": "checkpoints/original_seed42.pt",
  "checkpoint_mode": "original",
  "checkpoint_seed": 42,
  "sets": {
    "retain": {
      "n": 36000,
      "accuracy": 0.3244722222222222,
      "cross_entropy": 1.9313232844160815,
      "mean_confidence": 0.4406014084815979,
      "mean_entropy": 1.4591587781906128
    },
    "forget": {
      "n": 4000,
      "accuracy": 0.32325,
      "cross_entropy": 1.9484179794439995,
      "mean_confidence": 0.44270870089530945,
      "mean_entropy": 1.453965425491333
    },
    "auxiliary": {
      "n": 5000,
      "accuracy": 0.3144,
      "cross_entropy": 1.9729112781730394,
      "mean_confidence": 0.4

In [9]:
# STEP 9 — Remove ONLY the temporary one-epoch checkpoints/results
# Run this after the one-epoch test finishes successfully.

import shutil
from pathlib import Path

for folder_name in ["checkpoints", "results"]:
    folder = Path(folder_name)
    if folder.exists():
        for item in folder.iterdir():
            if item.is_dir():
                shutil.rmtree(item)
            else:
                item.unlink()

print("Temporary checkpoints/results cleared.")
print("The code, split file, and CIFAR-10 download are untouched.")


Temporary checkpoints/results cleared.
The code, split file, and CIFAR-10 download are untouched.


## Mount Google Drive before the real run

The real run can take a while. We will keep a persistent backup directory in Drive so your completed results are not left only in Colab's temporary `/content` storage.


In [10]:
# STEP 10 — Mount Google Drive and create the persistent Stage 1 folder

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

drive_stage1 = Path("/content/drive/MyDrive/PRISM-U/stage1")
drive_stage1.mkdir(parents=True, exist_ok=True)

print("Drive backup folder:", drive_stage1)


Mounted at /content/drive
Drive backup folder: /content/drive/MyDrive/PRISM-U/stage1


In [11]:
# STEP 11 — Optional: show GPU status immediately before real training

!nvidia-smi


Sat Oct 10 20:25:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Real Stage 1 run

The default project configuration trains **two ResNet-18 models for 40 epochs each**:

- original model on `retain + forget`
- honest retrained reference on `retain only`

Do not interrupt this cell unless necessary.


In [12]:
# STEP 12 — REAL STAGE 1 TRAINING

!python -m experiments.run_stage1 \
    --config configs/cifar10_stage1.yaml



$ /usr/bin/python3 -m data.splits --config configs/cifar10_stage1.yaml
Split already exists: splits/cifar10_seed42.json
{'target_train': 40000, 'forget': 4000, 'retain': 36000, 'auxiliary': 5000, 'validation': 5000, 'official_test': 10000}

$ /usr/bin/python3 -m models.train_target --mode original --config configs/cifar10_stage1.yaml
Training mode=original split=target_train samples=40000 device=cuda
epoch 001/40 train_loss=1.9226 train_acc=0.3027 val_loss=1.9664 val_acc=0.3160
epoch 002/40 train_loss=1.4053 train_acc=0.4817 val_loss=1.2504 val_acc=0.5482
epoch 003/40 train_loss=1.1248 train_acc=0.5956 val_loss=1.1351 val_acc=0.6016
epoch 004/40 train_loss=0.9198 train_acc=0.6727 val_loss=1.2016 val_acc=0.6174
epoch 005/40 train_loss=0.7686 train_acc=0.7333 val_loss=0.8009 val_acc=0.7262
epoch 006/40 train_loss=0.6576 train_acc=0.7721 val_loss=0.7883 val_acc=0.7220
epoch 007/40 train_loss=0.6036 train_acc=0.7922 val_loss=0.6633 val_acc=0.7772
epoch 008/40 train_loss=0.5565 train_acc=0

In [13]:
# STEP 13 — Back up everything important to Google Drive

import shutil
from pathlib import Path

drive_stage1 = Path("/content/drive/MyDrive/PRISM-U/stage1")
drive_stage1.mkdir(parents=True, exist_ok=True)

for folder_name in ["checkpoints", "results", "splits"]:
    src = Path(folder_name)
    dst = drive_stage1 / folder_name

    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f"Backed up {folder_name} -> {dst}")
    else:
        print(f"WARNING: {src} does not exist")

# Keep the exact config used as part of the experiment record
config_dst = drive_stage1 / "configs"
config_dst.mkdir(parents=True, exist_ok=True)
shutil.copy2("configs/cifar10_stage1.yaml", config_dst / "cifar10_stage1.yaml")

print("\nBackup complete.")


Backed up checkpoints -> /content/drive/MyDrive/PRISM-U/stage1/checkpoints
Backed up results -> /content/drive/MyDrive/PRISM-U/stage1/results
Backed up splits -> /content/drive/MyDrive/PRISM-U/stage1/splits

Backup complete.


In [14]:
# STEP 14 — Inspect the final Stage 1 metrics

import json
from pathlib import Path

def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

files_to_show = [
    "train_original_seed42.json",
    "train_retrain_seed42.json",
    "eval_original_seed42.json",
    "eval_retrain_seed42.json",
]

for name in files_to_show:
    path = Path("results") / name
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    if path.exists():
        print(json.dumps(load_json(path), indent=2))
    else:
        print("MISSING:", path)



train_original_seed42.json
{
  "mode": "original",
  "seed": 42,
  "train_split": "target_train",
  "train_samples": 40000,
  "epochs": 40,
  "best_val_acc": 0.9268,
  "test_loss": 0.2511528190135956,
  "test_acc": 0.9272,
  "training_seconds": 1834.544555902481,
  "checkpoint": "checkpoints/original_seed42.pt",
  "history": [
    {
      "epoch": 1,
      "train_loss": 1.9226126697540282,
      "train_acc": 0.3027,
      "val_loss": 1.9663838249206542,
      "val_acc": 0.316,
      "lr": 0.1
    },
    {
      "epoch": 2,
      "train_loss": 1.4053350547790526,
      "train_acc": 0.481675,
      "val_loss": 1.2503765995025635,
      "val_acc": 0.5482,
      "lr": 0.1
    },
    {
      "epoch": 3,
      "train_loss": 1.124836955833435,
      "train_acc": 0.595625,
      "val_loss": 1.1350520399093629,
      "val_acc": 0.6016,
      "lr": 0.1
    },
    {
      "epoch": 4,
      "train_loss": 0.9197779433250427,
      "train_acc": 0.672675,
      "val_loss": 1.2016474857330322,
      

In [15]:
# STEP 15 — Final integrity check

from pathlib import Path

required = [
    Path("checkpoints/original_seed42.pt"),
    Path("checkpoints/retrain_seed42.pt"),
    Path("results/train_original_seed42.json"),
    Path("results/train_retrain_seed42.json"),
    Path("results/eval_original_seed42.json"),
    Path("results/eval_retrain_seed42.json"),
    Path("results/probs_original_seed42.npz"),
    Path("results/probs_retrain_seed42.npz"),
    Path("splits/cifar10_seed42.json"),
]

missing = [str(p) for p in required if not p.exists()]

if missing:
    print("Stage 1 is NOT complete. Missing:")
    for p in missing:
        print(" -", p)
else:
    print("STAGE 1 COMPLETE ✅")
    print("All expected checkpoints, metrics, probabilities, and split files exist.")


STAGE 1 COMPLETE ✅
All expected checkpoints, metrics, probabilities, and split files exist.


In [16]:
# STEP 16 — Package the small result files for sharing

import shutil
from pathlib import Path

archive_base = "/content/prism_u_stage1_results"
archive_file = Path(archive_base + ".zip")

if archive_file.exists():
    archive_file.unlink()

shutil.make_archive(
    archive_base,
    "zip",
    root_dir="/content/PRISM-U-stage1",
    base_dir="results"
)

print("Created:", archive_file)
print("Size:", round(archive_file.stat().st_size / (1024 * 1024), 2), "MB")


Created: /content/prism_u_stage1_results.zip
Size: 4.41 MB


In [17]:
# STEP 17 — Download the result ZIP to your computer

from google.colab import files

files.download("/content/prism_u_stage1_results.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## After this notebook finishes

Keep the full backup in:

`MyDrive/PRISM-U/stage1/`

For the next PRISM-U stage, the important outputs are:

- `checkpoints/original_seed42.pt`
- `checkpoints/retrain_seed42.pt`
- `results/*.json`
- `results/*.npz`
- `splits/cifar10_seed42.json`

The next milestone is to implement an existing machine-unlearning method and create controlled incomplete-unlearning targets such as 25%, 50%, 75%, and full execution.
